In [ ]:
import os
import sys

# Detect Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Google Colab: {IN_COLAB}")

# In Colab, we may need to install pyprocar
if IN_COLAB:
    # Try to install pyprocar if not present
    !pip install pyprocar


In [ ]:
# @title PyProcar Configuration
# @markdown Set the path for the calculation directory (VASP or QE).
colab_path = "/content/calculation_dir" #@param {type:"string"}
local_path = "tests/io/vasp/" #@param {type:"string"}

data_path = colab_path if IN_COLAB else local_path

code = "vasp" #@param ["vasp", "qe"]

# @markdown ---
# @markdown ### Fermi Energy Settings
# @markdown Whether to shift bands by the Fermi energy.
use_fermi = True #@param {type:"boolean"}
# @markdown If you want to manually specify the Fermi energy, enter it here (or leave empty to auto-detect).
manual_fermi = "" #@param {type:"string"}

# @markdown ---
# @markdown ### Orbital Projection Settings
# @markdown Select which orbitals to project (for `bandsplot`).
plot_s_orbital = True #@param {type:"boolean"}
plot_p_orbital = True #@param {type:"boolean"}
plot_d_orbital = True #@param {type:"boolean"}

selected_orbitals = []
if plot_s_orbital:
    selected_orbitals.append('s')
if plot_p_orbital:
    selected_orbitals.append('p')
if plot_d_orbital:
    selected_orbitals.append('d')


In [ ]:
import pyprocar
import matplotlib.pyplot as plt
import numpy as np
from pyprocar.io.parser import Parser

def get_fermi(code, data_path, use_fermi, manual_fermi):
    if not use_fermi:
        # User opted to turn off Fermi shift
        return 0.0
        
    if manual_fermi.strip():
        try:
            return float(manual_fermi)
        except ValueError:
            print("Warning: manual_fermi is not a valid float. Will try auto-detect.")
            
    # Auto-detect Fermi energy
    try:
        parser = Parser(code=code, dirpath=data_path)
        return parser.ebs.efermi
    except Exception as e:
        print(f"Failed to auto-detect Fermi energy: {e}")
        return None


In [ ]:
# Get fermi energy for standard bandsplot
fermi_val = get_fermi(code, data_path, use_fermi, manual_fermi)

print("=== Standard bandsplot with selected orbitals ===")
if not selected_orbitals:
    # Plot plain if no orbitals selected
    pyprocar.bandsplot(code=code, dirname=data_path, mode="plain", fermi=fermi_val, show=True)
else:
    # Use overlay mode with selected orbitals for all atoms
    # We first need to find elements
    try:
        parser = Parser(code=code, dirpath=data_path)
        elements = np.unique(parser.structure.atoms)
        items = {el: selected_orbitals for el in elements}
        
        pyprocar.bandsplot(
            code=code, 
            dirname=data_path, 
            mode="overlay", 
            items=items, 
            fermi=fermi_val, 
            show=True
        )
    except Exception as e:
        print(f"Failed to run standard bandsplot with selected orbitals: {e}")

print("=== bandsplot_all (s, p, d for all elements) ===")
try:
    pyprocar.bandsplot_all(code=code, dirname=data_path, fermi=fermi_val, show=True)
except Exception as e:
    print(f"Failed to run bandsplot_all: {e}")
